# ModPlant-RTN Notebook Interface — Shared App Kernel

The widgets keep seed-driven linear/nonlinear recipe generation in Jupyter. Module data remains editable as `module_ops`, `_hc_data`, volumes and resources, while optimization, Connect/Disconnect lifecycle, energy/CO₂ accounting, validation, plan rows, explicit Master Recipe export, and Gantt rendering are delegated to the same service and visualization modules used by the desktop App.

## Section 1: Usage Notes

- Edit the Module dictionaries in Section 2 if required.
- Choose topology and App-equivalent objective/solver settings.
- Enter a seed or click **New Random Seed**.
- Click **Generate From Seed** to preview the General and Enriched control-flow graphs.
- Click **Generate & Solve** to invoke the shared ModPlant-RTN App kernel.
- The Gantt, dynamic plan rows, energy/CO₂ cost breakdown, Disconnect actions and validation therefore track future App-core changes automatically.

## Section 2: Setup, editable Module data, and shared state

Run this cell once. `module_ops` uses `(operation, parameter, usage_cost, energy_rate_kWh_s, co2_rate_kg_s, fixed_duration_s)`. Only Connect/Disconnect own fixed durations. The dictionaries are ordinary Python objects and may be changed directly before any solve.

In [ ]:
from pathlib import Path
import os
import importlib
import sys
from IPython.display import display

env_root = os.environ.get("MODPLANT_RTN_ROOT")
_candidate_roots = [Path(env_root).expanduser()] if env_root else []
_candidate_roots.extend([Path.cwd(), Path.cwd() / "RTN", Path.cwd().parent / "RTN"])
PROJECT_ROOT = next(
    (path.resolve() for path in _candidate_roots
     if (path / "scripts" / "rtn_notebook_ui.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run Jupyter from Module/RTN or set MODPLANT_RTN_ROOT.")
for _path in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

from sample_data import (
    sample_equipment_bindings,
    sample_module_ops,
)

# ---- EDITABLE MODPLANT INPUTS ----
# All default energy/CO₂ rates come from the App demo AAS model.
module_ops = sample_module_ops()

# (Module, number_of_inputs, number_of_outputs)
# Example: change HC30 from one to four output ports with ("HC30", 4, 4).
_hc_data = [
    ("HC10", 3, 3),
    ("HC20", 3, 3),
    ("HC30", 4, 1),
    ("HC40", 1, 2),
]

module_maximum_volume = {"HC10": [10.0], "HC20": [15.0], "HC30": [10.0], "HC40": [30.0]}
module_resources = {"HC10": ["A", 10.0], "HC20": ["B", 10.0], "HC30": ["C", 10.0]}
equipment_bindings = sample_equipment_bindings()

# Always derive physical ports after editing `_hc_data`.
module_interfaces = {
    hc: [("Input", f"{hc}_In{i}") for i in range(1, num_in + 1)]
        + [("Output", f"{hc}_Out{i}") for i in range(1, num_out + 1)]
    for hc, num_in, num_out in _hc_data
}

# Example override before constructing/running the UI:
# module_ops["HC10"][0] = ("Draining", 0.1, 3.0, 0.0611111111, 0.0244444444, 0.0)

import rtn_notebook_ui
importlib.reload(rtn_notebook_ui)
from rtn_notebook_ui import RTNNotebookUI

ui = RTNNotebookUI(
    project_root=PROJECT_ROOT,
    module_ops=module_ops,
    module_interfaces=module_interfaces,
    module_maximum_volume=module_maximum_volume,
    module_resources=module_resources,
    equipment_bindings=equipment_bindings,
).build()
PROJECT_ROOT

PosixPath('/Users/bowen/PycharmProjects/ModPlant-RTN')

## Section 3: App-equivalent objective and solver settings

These widgets map through the shared `RTNSettings.to_planner_config()` method, including revenue, time penalty, usage/energy/CO₂ weights, electricity price, Connect/Disconnect fallbacks, final-disconnect policy, time limit, workers and relative gap.

In [2]:
display(ui.settings_section)

## Section 4: Seed, Recipe, and Control-Flow Graphs

Generate the deterministic random recipe and inspect the original, pre-enrichment **General Recipe Control Flow** first, followed by the transformed **Enriched Recipe Control Flow**.

In [3]:
display(ui.context_section)

## Section 5: Shared App solve, Gantt, and dynamic plan table

Click **Generate & Solve**. Progress and elapsed time are updated while the desktop App service compiles RTN, runs CP-SAT, generates persistent Connect/Disconnect actions, accounts for energy/CO₂, and validates. No Master Recipe is generated until the explicit export action. The Gantt renderer and plan rows are the App implementations, not notebook copies.

In [4]:
display(ui.pipeline_section)

## Section 6: Independent Schedule Validation

Validation runs automatically after each solve and is displayed here. Click **Validate Latest Schedule** to run the independent validation again without solving again.

In [5]:
display(ui.validation_section)

## Section 7: Master Recipe Export

After a successful solve and validation, click **Export Master Recipe**. To export automatically after future solves, enable **Export automatically after solve**; it is off by default.

In [6]:
display(ui.export_section)

## Section 8: Re-running Safely

- Change the seed or settings and click **Generate & Solve** again.
- Widget outputs are cleared and replaced for every run.
- Restart the kernel only when changing installed dependencies or when a completely clean Python state is desired.